In [1]:
!pip install scapy numpy pandas joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 14.7 MB/s eta 0:00:00


In [8]:
from scapy.all import *
import numpy as np
import math
import joblib



# EXTRACT DHCP DISCOVER GAPS

def extract_gaps(pcap_file):

    packets = rdpcap(pcap_file)

    filtered_packets = []

    latest_packets = {}

    # STORE BOTH RAW + LOG GAPS
    raw_gaps = []
    log_gaps = []


    # KEEP ONLY LAST DISCOVER FOR SAME MAC + XID

    for pkt in packets:

        try:

            if not pkt.haslayer(BOOTP):
                continue

            if not pkt.haslayer(DHCP):
                continue

            # DHCP TYPE
            dhcp_type = None

            for op in pkt[DHCP].options:

                if isinstance(op, tuple):

                    if op[0] == 'message-type':

                        dhcp_type = int(op[1])

            # ONLY DISCOVER
            if dhcp_type != 1:
                continue

            mac = pkt.src

            xid = int(pkt[BOOTP].xid)

            key = (mac, xid)

            # OVERWRITE PREVIOUS
            latest_packets[key] = pkt

        except:
            pass

    # SORT FINAL PACKETS BY TIME

    filtered_packets = sorted(
        latest_packets.values(),
        key=lambda x: float(x.time)
    )


    # CALCULATE INTERPACKET GAPS

    prev_time = None

    for pkt in filtered_packets:

        current_time = float(pkt.time)

        if prev_time is not None:

            # RAW GAP
            raw_gap = current_time - prev_time

            # LOG GAP
            log_gap = math.log1p(raw_gap)

            # STORE BOTH
            raw_gaps.append(raw_gap)

            log_gaps.append(log_gap)

        prev_time = current_time

    return raw_gaps, log_gaps


# TRAIN MODEL

normal_raw_gaps, normal_log_gaps = extract_gaps(
    "dhcp_normall.pcap"
)

attack_raw_gaps, attack_log_gaps = extract_gaps(
    "starvation_changed_2.pcap"
)

# LEARN MEANS

normal_mean = np.mean(normal_log_gaps)

attack_mean = np.mean(attack_log_gaps)

threshold = (
    normal_mean + attack_mean
) / 2

# SAVE MODEL

model = {

    "normal_mean": normal_mean,

    "attack_mean": attack_mean,

    "threshold": threshold
}

joblib.dump(
    model,
    "dhcp_starvation_model.pkl"
)

# DISPLAY RESULTS

print("\n")
print("NORMAL RAW TIME GAPS : ")

print(normal_raw_gaps[:20])

print("\n")
print("NORMAL LOG TIME GAPS : ")

print(normal_log_gaps[:20])

print("\n")
print("ANOMALOUS RAW TIME GAPS : ")

print(attack_raw_gaps[:20])

print("\n")
print("ANOMALOUS LOG TIME GAPS : ")

print(attack_log_gaps[:20])

print("\n")
print("LEARNED VALUES : ")

print(f"\nNormal Mean Gap : {normal_mean}")

print(f"Attack Mean Gap : {attack_mean}")

print(f"Threshold       : {threshold}")

print("\nModel Saved Successfully")



NORMAL RAW TIME GAPS : 
[36.182048082351685, 42.70744013786316, 33.19957494735718, 33.910959005355835, 82.50538086891174, 152.6607940196991, 35.33188199996948, 124.77277612686157, 14.557483911514282, 38.01446795463562, 36.94461011886597, 151.20747303962708, 74.90094780921936, 50.558436155319214, 47.53066802024841]


NORMAL LOG TIME GAPS : 
[3.6158260663692485, 3.7775183424808394, 3.532213215551477, 3.5528007915318334, 4.424911071324396, 5.034747536806712, 3.592695647808245, 4.834476914860394, 2.744541803330171, 3.663932550531889, 3.6361274679016784, 5.025244544351169, 4.329429171928613, 3.942715847031471, 3.882195928421262]


ANOMALOUS RAW TIME GAPS : 
[5.152910947799683, 1.155113935470581, 1.1556611061096191, 1.1685409545898438, 1.1487901210784912, 1.1544768810272217, 1.1557860374450684, 1.1552369594573975, 1.1552329063415527, 1.1630840301513672, 1.146381139755249, 1.1545119285583496, 1.1421260833740234, 1.1449790000915527, 1.1595849990844727, 1.1637828350067139, 1.1636390686035156,

In [9]:
from scapy.all import *
import math
import joblib
import random


# LOAD MODEL
model = joblib.load(
    "dhcp_starvation_model.pkl"
)

print("\n")
print("MODEL LOADED")

# CLASSIFICATION FUNCTION

def classify_gap(raw_gap):

    log_gap = math.log1p(raw_gap)

    if log_gap < model["threshold"]:

        return 1

    else:

        return 0


# PROCESS PCAP

def process_pcap(pcap_file, label):

    packets = rdpcap(pcap_file)

    processed = []

    prev_time = None
    prev_mac = None
    prev_xid = None

    for pkt in packets:

        try:

            if not pkt.haslayer(BOOTP):
                continue

            if not pkt.haslayer(DHCP):
                continue

            # DHCP TYPE
            dhcp_type = None

            for op in pkt[DHCP].options:

                if isinstance(op, tuple):

                    if op[0] == 'message-type':

                        dhcp_type = int(op[1])

            # ONLY DISCOVER
            if dhcp_type != 1:
                continue

            current_time = float(pkt.time)

            current_mac = pkt.src

            current_xid = int(pkt[BOOTP].xid)

            # FIRST PACKET
            if prev_time is None:

                prev_time = current_time
                prev_mac = current_mac
                prev_xid = current_xid

                continue

            # SKIP RETRANSMISSIONS

            if current_mac == prev_mac and \
               current_xid == prev_xid:

                prev_time = current_time

                continue

            # GAP

            raw_gap = current_time - prev_time

            processed.append({

                "gap": raw_gap,

                "actual": label,

                "mac": current_mac,

                "xid": hex(current_xid)
            })

            # UPDATE
            prev_time = current_time
            prev_mac = current_mac
            prev_xid = current_xid

        except:
            pass

    return processed


# LOAD BOTH DATASETS

normal_data = process_pcap(
    "dhcp_normall.pcap",
    0
)

attack_data = process_pcap(
    "starvation_changed_2.pcap",
    1
)

# MERGE + SHUFFLE

all_data = normal_data + attack_data

random.shuffle(all_data)

print("\n")
print("RANDOMIZED TESTING STARTED")

# TESTING

y_true = []
y_pred = []

for sample in all_data:

    prediction = classify_gap(
        sample["gap"]
    )

    y_true.append(
        sample["actual"]
    )

    y_pred.append(
        prediction
    )

    label_name = \
        "ATTACK" if prediction == 1 \
        else "NORMAL"

    print("\n--")

    print(f"MAC        : {sample['mac']}")

    print(f"XID        : {sample['xid']}")

    print(f"Gap        : {sample['gap']:.6f}")

    print(f"Prediction : {label_name}")

    print("--")

Streaming output truncated to the last 5000 lines.
Prediction : ATTACK
--

--
MAC        : 02:9f:e5:a5:ea:57
XID        : 0xd5df6f01
Gap        : 1.146402
Prediction : ATTACK
--

--
MAC        : 02:cf:03:49:90:55
XID        : 0xdf8c42f5
Gap        : 1.152406
Prediction : ATTACK
--

--
MAC        : 02:6d:24:29:b0:a6
XID        : 0x8c586976
Gap        : 1.143439
Prediction : ATTACK
--

--
MAC        : 02:c5:6b:05:3a:d4
XID        : 0x5a4a0b9c
Gap        : 1.151942
Prediction : ATTACK
--

--
MAC        : 02:78:81:5b:9c:a0
XID        : 0x678349a2
Gap        : 1.146714
Prediction : ATTACK
--

--
MAC        : 02:62:2c:db:b5:11
XID        : 0xdc872239
Gap        : 1.159661
Prediction : ATTACK
--

--
MAC        : 02:ac:43:5c:88:b3
XID        : 0xd98d6d75
Gap        : 1.151311
Prediction : ATTACK
--

--
MAC        : 02:ff:be:63:0a:f2
XID        : 0x355545f9
Gap        : 1.149361
Prediction : ATTACK
--

--
MAC        : 02:4c:ff:99:8a:ce
XID        : 0xc15c16a5
Gap        : 1.152187
Prediction : 

In [10]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix


# METRICS

print("\n")
print("MODEL METRICS : ")

accuracy = accuracy_score(
    y_true,
    y_pred
)

print(f"\nAccuracy : {accuracy * 100:.2f}%")

print("\n")
print("CLASSIFICATION REPORT")
print("\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "NORMAL",
            "ATTACK"
        ]
    )
)

print("\n")
print("CONFUSION MATRIX")
print("\n")

print(
    confusion_matrix(
        y_true,
        y_pred
    )
)



MODEL METRICS : 

Accuracy : 100.00%


CLASSIFICATION REPORT


              precision    recall  f1-score   support

      NORMAL       1.00      1.00      1.00        15
      ATTACK       1.00      1.00      1.00      3413

    accuracy                           1.00      3428
   macro avg       1.00      1.00      1.00      3428
weighted avg       1.00      1.00      1.00      3428



CONFUSION MATRIX


[[  15    0]
 [   0 3413]]
